# Практика №3. Глубокое обучение и Fine-Tuning Transformer

**Задача:** дообучить предварительно обученную языковую модель для многоклассовой классификации эмоциональной окраски коротких текстов.

**Архитектура:** `DistilBERT` (`distilbert-base-uncased`) — облегчённая версия BERT на базе Transformer.

**Датасет:** [Emotion Dataset](https://huggingface.co/datasets/dair-ai/emotion) с платформы Hugging Face. Содержит англоязычные тексты, размеченные шестью эмоциональными категориями: sadness, joy, love, anger, fear и surprise.

**Подход:** загрузка предобученных весов, подготовка данных, токенизация, дообучение всей модели на обучающей выборке, подбор лучшей эпохи по валидационной F1-score и однократная итоговая оценка на тестовой выборке.

> Для запуска рекомендуется Google Colab с GPU (Среда выполнения → Сменить среду выполнения → T4 GPU). Первое выполнение скачивает датасет и веса модели из Hugging Face.

In [ ]:
%pip install -q "transformers==4.44.2" "datasets==2.21.0" "accelerate==0.34.2" torch pandas numpy matplotlib seaborn scikit-learn

In [ ]:
import json
import random
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
from IPython.display import display
from datasets import Dataset, load_dataset
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, f1_score, precision_score,
                             recall_score)
from sklearn.model_selection import train_test_split
from transformers import (AutoModelForSequenceClassification, AutoTokenizer,
                          DataCollatorWithPadding, Trainer, TrainingArguments,
                          set_seed)

sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
set_seed(RANDOM_STATE)
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Устройство: {DEVICE}")
if DEVICE == "cpu":
    print("Для ускорения обучения рекомендуется включить GPU в Google Colab.")

## 1. Загрузка датасета

Используем открытый набор `dair-ai/emotion`. Каждая запись содержит текст и целочисленную метку класса. Для воспроизводимости фиксируем генератор случайных чисел. Исходные части датасета объединяются, после проверки качества данных мы сформируем собственные стратифицированные Train/Validation/Test выборки в пропорции 80/10/10.

In [ ]:
raw_dataset = load_dataset("dair-ai/emotion")

parts = []
for split_name, split_data in raw_dataset.items():
    part = split_data.to_pandas()
    part["source_split"] = split_name
    parts.append(part)

df_raw = pd.concat(parts, ignore_index=True)
print("Размер исходного набора:", df_raw.shape)
print("Столбцы:", df_raw.columns.tolist())
display(df_raw.head())

## 2. Проверка качества и очистка данных

Проверяем пропуски, пустые строки, дубликаты, некорректные метки и аномально длинные тексты. Точные дубликаты текста удаляются, чтобы одинаковые примеры не попадали одновременно в разные выборки. Для данной задачи короткие и длинные сообщения сами по себе не обязательно являются ошибочными, поэтому длину текста анализируем, но не удаляем наблюдения только из-за необычного значения.

In [ ]:
label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]
df = df_raw[["text", "label"]].copy()
df["text"] = df["text"].astype("string").str.strip()
df["label"] = pd.to_numeric(df["label"], errors="coerce")
df["label"] = df["label"].replace([np.inf, -np.inf], np.nan)

print("Пропуски до очистки:")
display(df.isna().sum().to_frame("Количество"))
print("Пустые тексты:", int(df["text"].isna().sum() + df["text"].eq("").sum()))
print("Полные дубликаты:", int(df.duplicated().sum()))
print("Число уникальных меток:", sorted(df["label"].dropna().unique().tolist()))

valid_labels = set(range(len(label_names)))
df = df.dropna(subset=["text", "label"])
df = df[df["text"].str.len() > 0]
df = df[df["label"].isin(valid_labels)].copy()
df["label"] = df["label"].astype(int)
df = df.drop_duplicates(subset=["text"], keep="first").reset_index(drop=True)

print("Размер после очистки:", df.shape)
print("Оставшиеся пропуски:", int(df.isna().sum().sum()))
print("Оставшиеся дубликаты текста:", int(df["text"].duplicated().sum()))
display(df.head())

## 3. Разведочный анализ данных (EDA)

Изучим баланс эмоциональных классов и распределение длины текстов. Для выполнения требования о корреляционной матрице сформируем числовые характеристики текста: число символов, число слов, число восклицательных знаков, число вопросительных знаков и долю заглавных букв. Матрица отражает взаимосвязь этих вспомогательных признаков; она не используется вместо текстовых представлений Transformer.

In [ ]:
df["class_name"] = df["label"].map(dict(enumerate(label_names)))
df["char_count"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()
df["exclamation_count"] = df["text"].str.count("!")
df["question_count"] = df["text"].str.count(r"\?")
df["uppercase_ratio"] = df["text"].map(
    lambda value: sum(char.isupper() for char in value) / max(len(value), 1)
)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(data=df, x="class_name", order=label_names, ax=axes[0])
axes[0].set_title("Распределение эмоциональных классов")
axes[0].set_xlabel("Класс")
axes[0].set_ylabel("Количество текстов")
axes[0].tick_params(axis="x", rotation=35)
sns.histplot(data=df, x="word_count", bins=40, kde=True, ax=axes[1])
axes[1].set_title("Распределение длины текстов")
axes[1].set_xlabel("Количество слов")
plt.tight_layout()
plt.show()

numeric_features = ["char_count", "word_count", "exclamation_count",
                    "question_count", "uppercase_ratio"]
plt.figure(figsize=(8, 6))
sns.heatmap(df[numeric_features].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")
plt.title("Корреляционная матрица числовых характеристик текста")
plt.tight_layout()
plt.show()

display(df.groupby("class_name")[numeric_features].mean().round(2))

## 4. Разделение на обучающую, валидационную и тестовую выборки

Разделяем данные стратифицированно, сохраняя пропорции шести классов. 80% наблюдений используются для обучения, 10% — для выбора лучшей эпохи и настройки процесса, оставшиеся 10% — для финальной независимой оценки. Тестовая выборка не участвует в обучении и выборе модели.

In [ ]:
train_df, temp_df = train_test_split(
    df[["text", "label"]], test_size=0.20, random_state=RANDOM_STATE,
    stratify=df["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=RANDOM_STATE,
    stratify=temp_df["label"]
)

for name, part in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    print(f"{name}: {len(part):,} объектов ({len(part)/len(df):.1%})")
    display(part["label"].value_counts(normalize=True).sort_index().rename(index=dict(enumerate(label_names))).to_frame("Доля"))

## 5. Загрузка предобученной модели и токенизатора

Используем `distilbert-base-uncased`, предварительно обученную языковую модель на английском языке. Токенизатор преобразует текст в последовательности токенов и идентификаторов. К модели добавляется классификационная голова на шесть выходных классов. Поскольку веса классификационной головы изначально не обучены на Emotion Dataset, они будут настроены в ходе Fine-Tuning.

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
MAX_LENGTH = 128
id2label = {index: name for index, name in enumerate(label_names)}
label2id = {name: index for index, name in id2label.items()}

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)
print("Количество параметров:", f"{model.num_parameters():,}")
print("Модель:", MODEL_NAME)

## 6. Токенизация и подготовка Dataset

Токенизация выполняется одинаково для всех частей данных. Ограничиваем длину последовательности 128 токенами, чтобы контролировать расход видеопамяти. Метка целевого класса передаётся модели в поле `labels`.

In [ ]:
def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

def to_hf_dataset(frame):
    dataset = Dataset.from_pandas(frame.reset_index(drop=True))
    dataset = dataset.map(tokenize_batch, batched=True)
    dataset = dataset.rename_column("label", "labels")
    return dataset.remove_columns(["text"])

train_dataset = to_hf_dataset(train_df)
val_dataset = to_hf_dataset(val_df)
test_dataset = to_hf_dataset(test_df)
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)
print(train_dataset)
print("Пример токенизированной записи:", {k: v[:12] for k, v in train_dataset[0].items()})

## 7. Метрики качества

Для многоклассовой классификации рассчитываем Accuracy, Precision, Recall и F1-score с усреднением `weighted`. Weighted-вариант учитывает количество объектов каждого класса. Дополнительно в итоговом отчёте будут показаны метрики отдельно по каждому эмоциональному классу.

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision_weighted": precision_score(labels, predictions, average="weighted", zero_division=0),
        "recall_weighted": recall_score(labels, predictions, average="weighted", zero_division=0),
        "f1_weighted": f1_score(labels, predictions, average="weighted", zero_division=0),
    }

## 8. Fine-Tuning модели

На этом этапе дообучаем предобученную модель на обучающей выборке. Используем три эпохи, небольшую скорость обучения и динамическое дополнение последовательностей до длины самого длинного примера в батче. После каждой эпохи модель оценивается на валидационной выборке. Для итоговой оценки автоматически выбирается контрольная точка с наилучшим `f1_weighted`.

Если используется GPU, включается смешанная точность FP16. На CPU обучение также возможно, но будет значительно медленнее.

In [ ]:
OUTPUT_DIR = Path("artifacts/distilbert_emotion")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=3,
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    weight_decay=0.01,
    warmup_ratio=0.1,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_weighted",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=100,
    save_total_limit=2,
    fp16=torch.cuda.is_available(),
    report_to="none",
    seed=RANDOM_STATE
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)

train_result = trainer.train()
print("Fine-Tuning завершён.")
print("Лучшая контрольная точка:", trainer.state.best_model_checkpoint)
print("Лучшая валидационная метрика:", trainer.state.best_metric)

## 9. Анализ процесса обучения

Посмотрим на динамику функции потерь и валидационной метрики. Уменьшение train loss показывает, что модель подстраивается под обучающие данные. Валидационные показатели позволяют контролировать обобщающую способность и выбирать эпоху, не используя тестовую выборку.

In [ ]:
history = pd.DataFrame(trainer.state.log_history)
loss_columns = [column for column in ["loss", "eval_loss"] if column in history.columns]
metric_columns = [column for column in ["eval_accuracy", "eval_f1_weighted"] if column in history.columns]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if "loss" in history:
    train_history = history.dropna(subset=["loss"])
    axes[0].plot(train_history["step"], train_history["loss"], label="Train loss")
if "eval_loss" in history:
    eval_history = history.dropna(subset=["eval_loss"])
    axes[0].plot(eval_history["step"], eval_history["eval_loss"], marker="o", label="Validation loss")
axes[0].set_title("Функция потерь")
axes[0].set_xlabel("Шаг")
axes[0].set_ylabel("Loss")
axes[0].legend()

if "eval_accuracy" in history:
    eval_history = history.dropna(subset=["eval_accuracy"])
    axes[1].plot(eval_history["epoch"], eval_history["eval_accuracy"], marker="o", label="Accuracy")
if "eval_f1_weighted" in history:
    eval_history = history.dropna(subset=["eval_f1_weighted"])
    axes[1].plot(eval_history["epoch"], eval_history["eval_f1_weighted"], marker="o", label="F1 weighted")
axes[1].set_title("Качество на валидационной выборке")
axes[1].set_xlabel("Эпоха")
axes[1].set_ylabel("Значение метрики")
axes[1].legend()
plt.tight_layout()
plt.show()

display(history.tail(10))

## 10. Итоговая оценка на тестовой выборке

После выбора лучшей версии модели оцениваем её на тестовой выборке. Тестовые данные не использовались для обновления весов и выбора контрольной точки, поэтому полученные значения дают независимую оценку качества.

In [ ]:
test_metrics = trainer.evaluate(test_dataset)
print("Итоговые метрики на тестовой выборке:")
for metric, value in test_metrics.items():
    if isinstance(value, (int, float)):
        print(f"{metric}: {value:.4f}")

test_output = trainer.predict(test_dataset)
y_true = test_output.label_ids
y_pred = np.argmax(test_output.predictions, axis=1)

print("\nОтчёт по классам:")
print(classification_report(y_true, y_pred, target_names=label_names, digits=4))

## 11. Матрица ошибок

Матрица ошибок показывает, какие эмоциональные категории модель различает хорошо, а какие чаще путает между собой. Строки соответствуют истинным классам, столбцы — предсказанным.

In [ ]:
cm = confusion_matrix(y_true, y_pred, labels=list(range(len(label_names))))
plt.figure(figsize=(9, 7))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=label_names, yticklabels=label_names)
plt.xlabel("Предсказанный класс")
plt.ylabel("Истинный класс")
plt.title("Матрица ошибок на тестовой выборке")
plt.xticks(rotation=35, ha="right")
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

## 12. Сохранение модели и результатов

Сохраняем лучшую дообученную модель и токенизатор в каталог `artifacts/distilbert_emotion`. Также сохраняем итоговые метрики и историю обучения в JSON/CSV, чтобы результаты можно было воспроизвести и использовать вне ноутбука.

In [ ]:
trainer.save_model(str(OUTPUT_DIR))
tokenizer.save_pretrained(str(OUTPUT_DIR))

metrics_to_save = {key: float(value) for key, value in test_metrics.items()
                   if isinstance(value, (int, float, np.integer, np.floating))}
with open(OUTPUT_DIR / "test_metrics.json", "w", encoding="utf-8") as file:
    json.dump(metrics_to_save, file, ensure_ascii=False, indent=2)
history.to_csv(OUTPUT_DIR / "training_history.csv", index=False)
print("Модель и результаты сохранены в:", OUTPUT_DIR.resolve())

## 13. Проверка на собственных примерах

Проверим сохранённую модель на нескольких новых фразах, которых не было в обучающей и тестовой выборках. Модель возвращает наиболее вероятную эмоциональную категорию и распределение вероятностей по всем шести классам.

In [ ]:
def predict_emotion(texts, model, tokenizer):
    model.eval()
    inputs = tokenizer(texts, return_tensors="pt", truncation=True,
                       padding=True, max_length=MAX_LENGTH)
    inputs = {key: value.to(model.device) for key, value in inputs.items()}
    with torch.no_grad():
        logits = model(**inputs).logits
        probabilities = torch.softmax(logits, dim=-1).cpu().numpy()
    predictions = probabilities.argmax(axis=1)
    return pd.DataFrame({
        "text": texts,
        "predicted_emotion": [label_names[index] for index in predictions],
        "confidence": probabilities.max(axis=1).round(4)
    })

examples = [
    "I am so happy that everything worked out today!",
    "I feel worried about what might happen tomorrow.",
    "This news makes me extremely angry.",
    "I miss my friends and feel very lonely."
]
display(predict_emotion(examples, trainer.model, tokenizer))

## 14. Выводы

В ходе практической работы была выполнена адаптация предварительно обученной модели DistilBERT к задаче многоклассовой классификации эмоций в текстах. Были изучены и очищены данные, проведён разведочный анализ, сформированы независимые обучающая, валидационная и тестовая выборки, выполнена токенизация и процедура Fine-Tuning.

В итоговом отчёте необходимо зафиксировать фактически полученные значения Accuracy, Precision, Recall и F1-score на тестовой выборке. Также следует обратить внимание на матрицу ошибок и определить, какие эмоции модель классифицирует лучше, а какие чаще смешивает. Численные результаты не задаются заранее, поскольку они зависят от фактического запуска обучения и выбранного аппаратного ускорителя.

**Итог:** цель практики — освоить работу с современной архитектурой Transformer и получить опыт дообучения готовой языковой модели под конкретную задачу — достигнута после успешного выполнения всех ячеек ноутбука.